# Notebook 04 — Power BI Data Preparation
Generates 4 cleaned CSVs that Power BI will import directly.

In [7]:
from src.geospatial_utils import haversine_distance
import pathlib
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, '..')


PROC = pathlib.Path('..') / 'outputs' / 'processed_data'
print('Imports OK')

Imports OK


In [8]:
orders = pd.read_csv(PROC / 'olist_orders_dataset.csv')
items = pd.read_csv(PROC / 'olist_order_items_dataset.csv')
customers = pd.read_csv(PROC / 'olist_customers_dataset.csv')
sellers = pd.read_csv(PROC / 'olist_sellers_dataset.csv')
reviews = pd.read_csv(PROC / 'olist_order_reviews_dataset.csv')
products = pd.read_csv(PROC / 'olist_products_dataset.csv')
centroids = pd.read_csv(PROC / 'geo_centroids.csv')
print('All tables loaded.')

All tables loaded.


## 1. orders_geo.csv
Orders with seller/customer coordinates, distance, and delivery days.

In [9]:
# One price/seller per order
items_agg = items.groupby('order_id').agg(
    seller_id=('seller_id', 'first'),
    price=('price', 'sum'),
    freight_value=('freight_value', 'sum')
).reset_index()

# Seller coordinates
seller_coords = sellers.merge(
    centroids.rename(
        columns={'geolocation_zip_code_prefix': 'seller_zip_code_prefix'}),
    on='seller_zip_code_prefix', how='inner'
)[['seller_id', 'seller_state', 'lat', 'lng']].rename(columns={'lat': 'seller_lat', 'lng': 'seller_lng'})

# Customer coordinates
customer_coords = customers.merge(
    centroids.rename(
        columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix'}),
    on='customer_zip_code_prefix', how='inner'
)[['customer_id', 'customer_state', 'lat', 'lng']].rename(columns={'lat': 'customer_lat', 'lng': 'customer_lng'})

# Merge
orders_geo = (
    orders[orders['order_status'] == 'delivered']
    .merge(items_agg, on='order_id', how='left')
    .merge(seller_coords, on='seller_id', how='inner')
    .merge(customer_coords, on='customer_id', how='inner')
)

# Delivery days
for c in ['order_purchase_timestamp', 'order_delivered_customer_date']:
    orders_geo[c] = pd.to_datetime(orders_geo[c])
orders_geo['delivery_days'] = (
    orders_geo['order_delivered_customer_date'] - orders_geo['order_purchase_timestamp']).dt.days

# Distance
orders_geo['distance_km'] = haversine_distance(
    orders_geo['seller_lat'].values, orders_geo['seller_lng'].values,
    orders_geo['customer_lat'].values, orders_geo['customer_lng'].values
)

keep = ['order_id', 'order_purchase_timestamp', 'customer_state', 'seller_state',
        'seller_lat', 'seller_lng', 'customer_lat', 'customer_lng',
        'distance_km', 'delivery_days', 'price', 'freight_value']
orders_geo = orders_geo[keep].dropna(subset=['delivery_days', 'distance_km'])
orders_geo = orders_geo[orders_geo['delivery_days'] > 0]

orders_geo.to_csv(PROC / 'orders_geo.csv', index=False)
print(f'orders_geo.csv saved — {len(orders_geo):,} rows')

orders_geo.csv saved — 95,978 rows


## 2. state_summary.csv

In [10]:
review_map = reviews.groupby('order_id')['review_score'].mean()
orders_geo_r = orders_geo.copy()
orders_geo_r['review_score'] = orders_geo_r['order_id'].map(review_map)

state_summary = orders_geo_r.groupby('customer_state').agg(
    total_orders=('order_id', 'count'),
    avg_ticket=('price', 'mean'),
    avg_delivery_days=('delivery_days', 'mean'),
    avg_review_score=('review_score', 'mean'),
).round(2).reset_index()

state_summary.to_csv(PROC / 'state_summary.csv', index=False)
print(f'state_summary.csv saved — {len(state_summary)} states')
display(state_summary.sort_values('total_orders', ascending=False))

state_summary.csv saved — 27 states


,customer_state,total_orders,avg_ticket,avg_delivery_days,avg_review_score
25,SP,40393,125.18,8.30,4.25
18,RJ,12299,142.55,14.85,3.96
10,MG,11317,136.82,11.54,4.19
22,RS,5327,136.45,14.82,4.19
17,PR,4899,135.54,11.50,4.24
23,SC,3537,143.11,14.46,4.14
4,BA,3237,151.50,18.85,3.93
7,ES,1984,134.71,15.28,4.07
8,GO,1944,145.01,15.16,4.10
6,DF,1914,142.27,12.50,4.13


## 3. od_flows.csv

In [11]:
od_flows = (
    orders_geo
    .groupby(['seller_state', 'customer_state'])
    .agg(count=('order_id', 'count'), avg_distance_km=('distance_km', 'mean'))
    .round(2)
    .reset_index()
    .sort_values('count', ascending=False)
)

od_flows.to_csv(PROC / 'od_flows.csv', index=False)
print(f'od_flows.csv saved — {len(od_flows):,} O-D pairs')
display(od_flows.head(10))

od_flows.csv saved — 408 O-D pairs


,seller_state,customer_state,count,avg_distance_km
406,SP,SP,30675,150.05
399,SP,RJ,8131,440.92
391,SP,MG,7416,495.16
403,SP,RS,3582,906.14
398,SP,PR,3100,463.09
275,PR,SP,2920,469.95
167,MG,SP,2523,418.03
404,SP,SC,2320,561.99
385,SP,BA,2293,1388.14
152,MG,MG,1518,258.61


## 4. Verify Outputs

In [12]:
files = ['orders_geo.csv', 'state_summary.csv',
         'od_flows.csv', 'seller_clusters.csv']
for f in files:
    p = PROC / f
    if p.exists():
        df = pd.read_csv(p)
        print(f'  {f}: {len(df):,} rows x {len(df.columns)} cols')
    else:
        print(f'  MISSING: {f}')

  orders_geo.csv: 95,978 rows x 12 cols
  state_summary.csv: 27 rows x 5 cols
  od_flows.csv: 408 rows x 4 cols
  seller_clusters.csv: 3,088 rows x 6 cols
